In [1]:
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

np.random.seed(42)
n = 300
time_idx = np.arange(n)
trend = time_idx * 0.05 
noise = np.random.normal(0, 1, n)
y = trend + noise

X = pd.DataFrame({"time_idx": time_idx})
y = pd.Series(y)

# Random KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
kf_scores = []
for train_idx, test_idx in kf.split(X):
    model = LinearRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    kf_scores.append(mean_absolute_error(y.iloc[test_idx], pred))

# TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
ts_scores = []
for train_idx, test_idx in tscv.split(X):
    model = LinearRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    ts_scores.append(mean_absolute_error(y.iloc[test_idx], pred))

print("KFold MAE:", np.mean(kf_scores), kf_scores)
print("TimeSeriesSplit MAE:", np.mean(ts_scores), ts_scores)

KFold MAE: 0.7782006469686742 [0.6315518648342486, 0.8435102790214636, 0.8189859446008723, 0.8423635879749577, 0.7545915584118288]
TimeSeriesSplit MAE: 0.847379259588837 [1.0208729937250711, 0.7960750524442105, 0.7134931954098209, 0.8435420557723657, 0.8629130005927166]


In [2]:
import duckdb
import pandas as pd

duckdb.sql("CREATE VIEW duolingo_flagship AS SELECT * FROM read_csv_auto('../../data/duolingo_flagship_v4.csv')")
df = duckdb.sql("SELECT * FROM duolingo_flagship").df()

df.shape

(16382, 17)

In [3]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits = 5)

In [4]:
for train_idx, test_idx in gkf.split(df, groups=df["user_id"]):
    print("train:", len(train_idx), "test:", len(test_idx))

train: 13105 test: 3277
train: 13105 test: 3277
train: 13106 test: 3276
train: 13106 test: 3276
train: 13106 test: 3276


In [5]:
for train_idx, test_idx in gkf.split(df, groups=df["user_id"]):
    train_users = set(df.iloc[train_idx]["user_id"])
    test_users = set(df.iloc[test_idx]["user_id"])
    overlap = train_users & test_users
    print("number of common users:", len(overlap))

number of common users: 0
number of common users: 0
number of common users: 0


number of common users: 0
number of common users: 0


In [6]:
import numpy as np

# sorted() makes the split independent of row order in the CSV.
# Without it, any later step that reorders rows (the SQL LEFT JOIN in part 14
# does exactly this) silently produces a different split from the same seed.
unique_users = np.array(sorted(df["user_id"].unique()))

rng = np.random.default_rng(42)
rng.shuffle(unique_users)

n_test_users = int(len(unique_users) * 0.15)
test_users = set(unique_users[:n_test_users])
cv_users = set(unique_users[n_test_users:])

df_test = df[df["user_id"].isin(test_users)]
df_cv = df[df["user_id"].isin(cv_users)]

print("hold-out test:", df_test.shape, "| CV pool:", df_cv.shape)
print("test users:", len(test_users), "| CV users:", len(cv_users))
print("number of common users:", len(test_users & cv_users))

hold-out test: (1944, 17) | CV pool: (14438, 17)
test users: 375 | CV users: 2125
number of common users: 0


In [7]:
# Persist the split so every later step reads it instead of re-deriving it.
# It is keyed on user_id alone, so it stays valid across dataset versions
# (v4/v5/v6 all contain the same 2,500 users).
split_users = pd.DataFrame({
    "user_id": sorted(test_users) + sorted(cv_users),
    "split": ["test"] * len(test_users) + ["cv"] * len(cv_users),
})
split_users.to_csv("../../data/split_users.csv", index=False)

# Reload and verify the saved file reproduces the split exactly.
check = pd.read_csv("../../data/split_users.csv")
check_test = set(check.loc[check["split"] == "test", "user_id"])
check_cv = set(check.loc[check["split"] == "cv", "user_id"])

assert check_test == test_users
assert check_cv == cv_users
assert not (check_test & check_cv)
assert check_test | check_cv == set(df["user_id"])

print(split_users["split"].value_counts().to_string())
print("split file verified:", split_users.shape)

split
cv      2125
test     375
split file verified: (2500, 2)


## Split strategy

Grain: one row is one practice session for a (user, lexeme) pair. Sessions aren't independent,same user shows up in multiple rows.

Goal is predicting p_recall for a genuinely new user, not a known user's future session. That's the part that decides everything else.

Went with GroupKFold on user_id, no time component. Group structure is real (users repeat), so GroupKFold is needed. Checked this both in the mini lab and on the actual data, 0 overlapping users across all 5 folds. Didn't add a temporal split on top since the target scenario is "new user," not "future session" when a user shows up doesn't matter, only that their user_id never leaks across train and test.

Held out 15% of users as a final test set, untouched until evaluation (375 users, 1,944 rows). The rest (2,125 users, 14,438 rows) is the CV pool for GroupKFold during model development.

## The split is saved, not re-derived

First version of this notebook computed the split and never wrote it down, and parts 12/13 just re-ran the same lines. That looked fine because the seed was fixed, but the code shuffled `df["user_id"].unique()`, which returns users in *row order*. The SQL `LEFT JOIN` in part 14 reorders rows, so from v5 onward the identical seed produced a different split, only 70 of the 375 hold-out users survived. Baselines measured before that point were no longer measured on the same data.

Two fixes, both in this notebook:

1. `sorted()` the users before shuffling, so row order can't influence the result. Verified this now yields the same split from v4, v5 and v6.
2. Write the split to `data/split_users.csv` and have every later step load it. Because it's keyed on `user_id` only, it stays valid no matter which dataset version a notebook reads.